In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found.")

    print("✅ API Key loaded successfully.")

except Exception as e:
    print(f"❌ Error: {e}")
    GROQ_API_KEY = None

✅ API Key loaded successfully.


In [3]:
from groq import Groq
import re

client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.1-8b-instant"
TEMPERATURE = 0
MAX_TOKENS = 300

In [4]:
def ask_llm(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"API Error: {e}"

In [5]:
problem = (
    "A farmer has 15 cows. He buys 7 more cows. "
    "Then he sells half of his cows. "
    "How many cows does he have left?"
)

direct_prompt = problem

cot_prompt = (
    problem +
    "\n\nLet's think step by step. "
    "Show each calculation clearly before giving the final answer."
)

In [6]:
direct_response = ask_llm(direct_prompt)

cot_response = ask_llm(cot_prompt)

In [7]:
import re

def extract_answer(text):
    numbers = re.findall(r"\d+", text)

    if numbers:
        return numbers[-1]

    return "Not Found"

direct_answer = extract_answer(direct_response)
cot_answer = extract_answer(cot_response)

EXPECTED = "11"

In [8]:
print("="*80)
print("DIRECT PROMPT")
print("="*80)

print("\nPrompt:")
print(direct_prompt)

print("\nResponse:")
print(direct_response)

print("\nExtracted Answer:", direct_answer)

print("\n")

print("="*80)
print("CHAIN-OF-THOUGHT PROMPT")
print("="*80)

print("\nPrompt:")
print(cot_prompt)

print("\nResponse:")
print(cot_response)

print("\nExtracted Answer:", cot_answer)

DIRECT PROMPT

Prompt:
A farmer has 15 cows. He buys 7 more cows. Then he sells half of his cows. How many cows does he have left?

Response:
To find out how many cows the farmer has left, we need to follow the sequence of events:

1. The farmer starts with 15 cows.
2. He buys 7 more cows, so now he has 15 + 7 = 22 cows.
3. He sells half of his cows. Half of 22 is 11, so he sells 11 cows.
4. After selling 11 cows, he is left with 22 - 11 = 11 cows.

So, the farmer has 11 cows left.

Extracted Answer: 11


CHAIN-OF-THOUGHT PROMPT

Prompt:
A farmer has 15 cows. He buys 7 more cows. Then he sells half of his cows. How many cows does he have left?

Let's think step by step. Show each calculation clearly before giving the final answer.

Response:
To find out how many cows the farmer has left, we'll break down the problem into steps.

**Step 1: Initial number of cows**
The farmer starts with 15 cows.

**Step 2: Buying more cows**
He buys 7 more cows. To find the total number of cows after bu

In [9]:
print("="*80)
print("OPTIONAL TASK DECOMPOSITION")
print("="*80)

step1 = ask_llm(
    "A farmer has 15 cows and buys 7 more cows. "
    "How many cows does he have now?"
)

step2 = ask_llm(
    "A farmer has 22 cows and sells half of them. "
    "How many cows are left?"
)

print("\nStep 1:")
print(step1)

print("\nStep 2:")
print(step2)

OPTIONAL TASK DECOMPOSITION

Step 1:
To find the total number of cows the farmer has now, we need to add the initial number of cows (15) to the number of cows he bought (7).

15 (initial cows) + 7 (new cows) = 22

So, the farmer now has 22 cows.

Step 2:
To find out how many cows are left, we need to calculate half of 22. 

Half of 22 is 22 / 2 = 11.

So, the farmer sells 11 cows. To find out how many cows are left, we subtract 11 from 22.

22 - 11 = 11.

Therefore, the farmer has 11 cows left.


In [10]:
print("="*80)
print("ANALYSIS")
print("="*80)

print(f"\nExpected Answer : {EXPECTED}")
print(f"Direct Answer   : {direct_answer}")
print(f"CoT Answer      : {cot_answer}")

print("\nComparison")

if direct_answer == EXPECTED:
    print("- Direct prompt produced the correct answer.")
else:
    print("- Direct prompt did not produce the expected answer.")

if cot_answer == EXPECTED:
    print("- Chain-of-Thought produced the correct answer.")
else:
    print("- Chain-of-Thought did not produce the expected answer.")

print("\nConclusion")
print("Chain-of-Thought prompting encourages the model to reason")
print("through each step before producing the final answer.")
print("This generally improves reliability and makes the reasoning")
print("easy to understand.")

ANALYSIS

Expected Answer : 11
Direct Answer   : 11
CoT Answer      : 11

Comparison
- Direct prompt produced the correct answer.
- Chain-of-Thought produced the correct answer.

Conclusion
Chain-of-Thought prompting encourages the model to reason
through each step before producing the final answer.
This generally improves reliability and makes the reasoning
easy to understand.
